In [142]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import time

In [138]:
df_train = pd.read_csv("dados/nba_treino.csv")
df_test = pd.read_csv("dados/nba_teste.csv")

target_knn = 'TARGET_5Yrs'

X_train = df_train.drop(columns=target_knn)
X_test = df_test.drop(columns=target_knn)
y_train = df_train[target_knn]
y_test = df_test[target_knn]

In [147]:
def euclidian_distance(p1, p2):
    array1 = np.array(p1)
    array2 = np.array(p2)

    return np.sum((array1 - array2)**2)

def sort_by_distance(center, df, labels):
    dists_and_labels = [] 
    
    for i, point in enumerate(df.to_numpy()):
        dist = euclidian_distance(center, point)
        dists_and_labels.append((dist, labels[i])) # o primeiro elemento é a distância entre o ponto e center, o segundo é a label do ponto

    dists_and_labels.sort(key=lambda x: x[0])

    labels_sorted = pd.DataFrame([ l[1] for l in dists_and_labels ]) # só as labels, sem a distância

    return labels_sorted

def knn_point(X_train, y_train, point, k):
    labels_by_dist = sort_by_distance(point, X_train, y_train)
    neighbors = labels_by_dist[:k]

    neighbor_targets = neighbors.value_counts() # o número de ocorrências de cada classe em neighbors

    return neighbor_targets.nlargest(1).index[0][0] # a classe de maior ocorrência

def knn_dataset(X_train, y_train, X_test, k):
    start = time.time()

    predictions = X_test.apply(lambda x: knn_point(X_train, y_train, x, k), axis=1)

    end = time.time()
    time_elapsed = round(end-start, 2)
    
    print("KNN feito em", time_elapsed, "segundos!")

    return predictions

In [148]:
predictions_knn = knn_dataset(X_train, y_train, X_test, 5)

KNN feito em 3.74 segundos!


In [145]:
from sklearn import metrics

metrics.precision_score(y_test, predictions_knn)

0.6994219653179191